# Customer Review Analysis System

This notebook follows the Piton Technology technical case flow: data loading, cleaning, EDA, text preprocessing, sentiment modeling, hyperparameter optimization, fuzzy reliability scoring, error analysis, and business complaint summary.

## 1. Imports

In [ ]:
from pathlib import Path
import sys

import joblib
import pandas as pd
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import DEFAULT_RAW_PATH, DEFAULT_CLEAN_PATH, create_clean_sample
from src.eda import get_basic_statistics, get_class_distribution, get_missing_summary, save_eda_plots
from src.preprocessing import demonstrate_preprocessing_steps, preprocess_text
from src.train import build_logistic_pipeline, build_random_forest_pipeline, run_grid_search
from src.evaluate import create_error_analysis_table, evaluate_model, plot_confusion_matrix, save_model_comparison
from src.fuzzy_system import apply_reliability_scores, compute_reliability_score
from src.complaint_summary import get_top_complaints

FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
MODELS_DIR = PROJECT_ROOT / "outputs" / "models"
REPORTS_DIR = PROJECT_ROOT / "outputs" / "reports"
for path in (FIGURES_DIR, MODELS_DIR, REPORTS_DIR):
    path.mkdir(parents=True, exist_ok=True)

## 2. Dataset Loading

The project uses the local Kaggle TSV file under `data/raw/`. If a clean sample already exists, it is loaded directly.

In [ ]:
if DEFAULT_CLEAN_PATH.exists():
    df = pd.read_csv(DEFAULT_CLEAN_PATH, parse_dates=["review_date"])
else:
    df = create_clean_sample(DEFAULT_RAW_PATH, DEFAULT_CLEAN_PATH, sample_size=30_000, random_state=42)

df.head()

## 3. Missing Value Analysis and Column Overview

In [ ]:
print("Shape:", df.shape)
display(pd.DataFrame({"dtype": df.dtypes.astype(str), "non_null": df.notna().sum()}))
display(get_missing_summary(df))

## 4. EDA Visualizations

The following code saves the required EDA plots to `outputs/figures/`.

In [ ]:
saved_plots = save_eda_plots(df, FIGURES_DIR)
saved_plots

## 5. Descriptive Statistics and Class Imbalance

Positive reviews are often dominant in e-commerce data. Because of this, macro F1 and weighted F1 are more informative than accuracy alone.

In [ ]:
display(get_basic_statistics(df))
display(get_class_distribution(df))

## 6. Text Preprocessing Example

The pipeline applies lowercasing, cleaning, stop word removal, and lemmatization.

In [ ]:
example_text = df["review_text"].dropna().iloc[0]
pd.DataFrame(demonstrate_preprocessing_steps(example_text).items(), columns=["step", "text"])

## 7. TF-IDF vs Bag-of-Words

Bag-of-Words counts word occurrences. TF-IDF weights words by importance across the corpus and reduces the impact of very common terms. This project uses TF-IDF because sparse high-dimensional text features usually work well with linear models such as Logistic Regression.

## 8. Train/Test Split

The split is stratified by sentiment and uses `random_state=42`.

In [ ]:
df_model = df.copy()
df_model["processed_text"] = df_model["review_text"].fillna("").astype(str).map(preprocess_text)
df_model = df_model[df_model["processed_text"].str.strip().ne("")]

x_train, x_test, y_train, y_test = train_test_split(
    df_model["processed_text"],
    df_model["sentiment"],
    test_size=0.2,
    stratify=df_model["sentiment"],
    random_state=42,
)
metadata = df_model.loc[y_test.index]
x_train.shape, x_test.shape

## 9. Baseline Logistic Regression

In [ ]:
logistic_model = build_logistic_pipeline()
logistic_model.fit(x_train, y_train)
logistic_metrics = evaluate_model(logistic_model, x_test, y_test, "Logistic Regression", FIGURES_DIR)
print(logistic_metrics["classification_report"])

## 10. Random Forest

In [ ]:
random_forest_model = build_random_forest_pipeline()
random_forest_model.fit(x_train, y_train)
rf_metrics = evaluate_model(random_forest_model, x_test, y_test, "Random Forest", FIGURES_DIR)
print(rf_metrics["classification_report"])

## 11. Model Comparison

Logistic Regression is expected to be a strong candidate because TF-IDF produces sparse high-dimensional features. Random Forest is included as a comparison and may be slower or weaker on this representation.

In [ ]:
comparison = save_model_comparison(
    [logistic_metrics, rf_metrics],
    REPORTS_DIR / "model_comparison.csv",
)
comparison

## 12. GridSearchCV Optimization

The optimized Logistic Regression pipeline uses `scoring="f1_macro"` and `cv=3`.

In [ ]:
search = run_grid_search(x_train, y_train)
optimized_model = search.best_estimator_
optimized_metrics = evaluate_model(optimized_model, x_test, y_test, "Optimized Logistic Regression", None)
print(search.best_params_)
print(optimized_metrics["classification_report"])

pd.DataFrame(search.cv_results_).to_csv(REPORTS_DIR / "grid_search_results.csv", index=False)
pd.DataFrame([search.best_params_]).to_csv(REPORTS_DIR / "best_grid_params.csv", index=False)
pd.DataFrame([
    {"model": "Baseline Logistic Regression", "accuracy": logistic_metrics["accuracy"], "f1_macro": logistic_metrics["f1_macro"], "f1_weighted": logistic_metrics["f1_weighted"]},
    {"model": "Optimized Logistic Regression", "accuracy": optimized_metrics["accuracy"], "f1_macro": optimized_metrics["f1_macro"], "f1_weighted": optimized_metrics["f1_weighted"]},
]).to_csv(REPORTS_DIR / "optimized_model_metrics.csv", index=False)

## 13. Best Model Confusion Matrix and Persistence

In [ ]:
candidate_metrics = [logistic_metrics, rf_metrics, optimized_metrics]
best_metrics = max(candidate_metrics, key=lambda item: item["f1_macro"])
best_model = {
    "Logistic Regression": logistic_model,
    "Random Forest": random_forest_model,
    "Optimized Logistic Regression": optimized_model,
}[best_metrics["model"]]

plot_confusion_matrix(
    y_test,
    best_metrics["y_pred"],
    title=f"Confusion Matrix - Best Model ({best_metrics['model']})",
    output_path=FIGURES_DIR / "confusion_matrix_best_model.png",
)
joblib.dump(best_model, MODELS_DIR / "best_sentiment_model.joblib")

## 14. Fuzzy Logic Reliability Scoring

The fuzzy score uses rating, review length, and review age. It is a transparent business confidence layer, not an absolute truth signal.

In [ ]:
sample_review = df_model.iloc[0]
reliability_score = compute_reliability_score(
    rating=sample_review["star_rating"],
    review_length=sample_review["review_length"],
    review_age_days=sample_review["review_age_days"],
)
print("Reliability score:", reliability_score)

reliability_sample = apply_reliability_scores(df_model.head(500))
reliability_sample.to_csv(REPORTS_DIR / "reliability_sample.csv", index=False)
reliability_sample[["star_rating", "review_length", "review_age_days", "reliability_score"]].head()

## 15. Error Analysis

The table below identifies five misclassified examples and adds a short interpretation for each error.

In [ ]:
error_table = create_error_analysis_table(best_model, x_test, y_test, metadata, top_n=5)
error_table.to_csv(REPORTS_DIR / "error_analysis.csv", index=False)
error_table

## 16. Business Complaint Summary

This function supports category and date filtering. Historical data may not contain current-week reviews, so the same logic can be used with any date range.

In [ ]:
complaints = get_top_complaints(df_model, top_n=10)
complaints.to_csv(REPORTS_DIR / "top_complaints.csv", index=False)
complaints

## 17. Conclusion

This notebook builds a complete local pipeline without fabricating metrics. After execution, generated figures, model files, comparison tables, optimized metrics, reliability samples, error analysis, and complaint summaries are saved under `outputs/`.